https://www.youtube.com/watch?v=CXXl9CFY5ZU


In [1]:
import os
import ee
import geemap
from dotenv import load_dotenv

from tools.Input_values.satellites import sentinel2_harmonized
from tools.Input_values.locaties import select_naturenreserves, build_location_bound
from tools.data_cleansing.sentinel2 import add_local_cloud_cover, scl_cloud_removal_20m, scale_reflectance
from tools.calculate_spectral_indices.sentinel2 import calc_ndvi


# Authenticate only once if needed
# ee.Authenticate(auth_mode='localhost', force=True)

load_dotenv()

project_code_ee = os.getenv("PROJECT_CODE_EE")

if not project_code_ee:
    raise RuntimeError("PROJECT_CODE_EE not found")

ee.Initialize(project=project_code_ee)

# Maak kaart
Map = geemap.Map()
Map.add_basemap("satellite")

# -----------------------------
# 1. Natuurgebied selecteren
# -----------------------------
natuurgebieden = select_naturenreserves(["Nieuwkoopse Plassen & de Haeck"])

# Controleer het gebied op de kaart
Map.centerObject(natuurgebieden, 11)
Map.addLayer(natuurgebieden, {"color": "red"}, "Natuurgebied")

# -----------------------------
# 4. Sentinel-2 collectie maken
# -----------------------------
s2 = (
    ee.ImageCollection(sentinel2_harmonized)
    .filterDate("2025-05-01", "2025-08-01")
    .filterBounds(natuurgebieden)
    .map(add_local_cloud_cover(natuurgebieden, coverage = "bounds"))
    .filter(ee.Filter.lte("LOCAL_CLOUD_COVER", 30))
    # .map(probability_cloud_removal_20m(35))
    # .map(scl_cloud_removal_20m(remove_unclassified=False, bloat_n_meters= 50))
    .map(scale_reflectance)
    .map(calc_ndvi)
)

# -----------------------------
# 5. Maak 1 NDVI eindbeeld
#    Hier nemen we de mediaan
#    van alle NDVI-beelden in de tijd
# -----------------------------
ndvi_gebied = s2.select("NDVI").median().clip(build_location_bound(locations_of_interest=natuurgebieden, mode="geometry"))

# -----------------------------
# 6. Visualisatie
# -----------------------------
ndvi_vis = {"min": 0.0, "max": 1.0, "palette": ["brown", "yellow", "green"]}

Map.addLayer(ndvi_gebied, ndvi_vis, "NDVI natuurgebied")

# Optioneel: satellietbeeld ter referentie
rgb = s2.median().clip(build_location_bound(locations_of_interest=natuurgebieden, mode="geometry"))
rgb_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}
Map.addLayer(rgb, rgb_vis, "RGB median")

# median() → robust to outliers (very popular for vegetation)
# sum() → total over time
# min() → lowest value per pixel
# max() → highest value per pixel
# mode() → most frequent value (useful for categorical data)
# stdDev() → variability over time
# variance() → statistical spread
# reduce(ee.Reducer...) → fully custom reduction

# clip(geometry) → hard crop to boundary
# clipToCollection(fc) → clip to multiple features
# updateMask(mask) → soft masking (very important alternative to clip)
# selfMask() → mask zeros automatically

Map

Map(center=[52.142152501965185, 4.804283214205478], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import pandas as pd
from IPython.display import display
from tools.data_analysis.geemap_data import (
    describe_image_properties,
    image_collection_to_metadata_table,
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(describe_image_properties(s2, "sentinel2_sr_harmonized"))

,property_name,property_category,example_value,value_type,description
0,AOT_RETRIEVAL_ACCURACY,dataset,0,int,Nauwkeurigheid van het aerosol optical thickness model.
1,AOT_RETRIEVAL_METHOD,dataset,SEN2COR_DDV,str,Geen omschrijving beschikbaar.
2,BOA_ADD_OFFSET_B1,dataset,-1000,int,Geen omschrijving beschikbaar.
3,BOA_ADD_OFFSET_B10,dataset,-1000,int,Geen omschrijving beschikbaar.
4,BOA_ADD_OFFSET_B11,dataset,-1000,int,Geen omschrijving beschikbaar.
...,...,...,...,...,...
101,system:id,system,None,NoneType,Geen omschrijving beschikbaar.
102,system:index,system,None,NoneType,Unieke index of identifier van het beeld binnen de dataset.
103,system:time_end,system,None,NoneType,Geen omschrijving beschikbaar.
104,system:time_start,system,None,NoneType,"Tijdstip waarop de opname is gemaakt, opgeslagen als Unix-tijd in milliseconden."


In [ ]:
image_metadata = image_collection_to_metadata_table(
    collection=s2,
    locations_of_interest=natuurgebieden,
    coverage="bounds",
    buffer_meters=0,
    bands=["NDVI"],
    include_properties=[
        "CLOUDY_PIXEL_PERCENTAGE",
        "LOCAL_CLOUD_COVER",
        "MGRS_TILE",
    ],
    scale=10,
)
image_metadata

,CLOUDY_PIXEL_PERCENTAGE,LOCAL_CLOUD_COVER,MGRS_TILE,NDVI_mean,NDVI_median,NDVI_valid_fraction,datetime,image_id,region_buffer_meters,region_mode
0,0.042023,0.002068,31UFT,0.679502,0.773618,0.950626,2025-05-01 10:47:02,20250501T104041_20250501T104252_T31UFT,0,bounds
1,33.004585,0.007742,31UFT,0.657613,0.730479,0.846586,2025-05-03 10:46:53,20250503T103701_20250503T103937_T31UFT,0,bounds
2,0.028323,0.004091,31UFT,0.634187,0.695325,0.880842,2025-05-11 10:46:58,20250511T103641_20250511T103655_T31UFT,0,bounds
3,0.028817,0.007067,31UFT,0.628143,0.699229,0.834732,2025-05-13 10:46:54,20250513T104041_20250513T104446_T31UFT,0,bounds
4,7.974831,20.770833,31UFT,0.571863,0.628920,1.000000,2025-05-16 10:56:52,20250516T104651_20250516T105456_T31UFT,0,bounds
5,5.482537,3.156230,31UFT,0.638297,0.722648,1.000000,2025-05-19 10:56:38,20250519T104619_20250519T104812_T31UFT,0,bounds
6,0.647622,0.009815,31UFT,0.692260,0.775377,0.801339,2025-06-12 10:46:55,20250612T103701_20250612T103656_T31UFT,0,bounds
7,23.424965,3.317816,31UFT,0.636407,0.691466,1.000000,2025-06-13 10:56:57,20250613T104641_20250613T104639_T31UFT,0,bounds
8,27.741125,0.000000,31UFT,0.641905,0.722643,1.000000,2025-06-18 10:56:37,20250618T104619_20250618T104619_T31UFT,0,bounds
9,2.987466,0.000000,31UFT,0.645739,0.742362,0.888698,2025-06-30 10:47:01,20250630T104041_20250630T104234_T31UFT,0,bounds
